# geeViz Google Maps Platform Examples

This notebook demonstrates `geeViz.googleMapsLib` — 24 functions for ground-truthing remote sensing analysis using Google Maps Platform APIs.

**Requirements:**
- `pip install geeViz`
- A `.env` file in the geeViz directory with:
  ```
  GEMINI_API_KEY=your_key_here
  GOOGLE_MAPS_PLATFORM_API_KEY=your_key_here
  ```
- For segmentation: `pip install geeViz[segmentation]`

| Section | APIs Used |
|---|---|
| 1. Geocoding | Geocoding, Reverse Geocoding, Address Validation |
| 2. Street View | Street View Static, Panorama Stitching |
| 3. Places | Places (New), Place Photos |
| 4. Environment | Elevation, Air Quality, Solar, Timezone |
| 5. AI Analysis | Gemini Interpretation, Object Detection |
| 6. Segmentation | SegFormer Semantic Segmentation |

In [ ]:
import geeViz.googleMapsLib as gm
from IPython.display import display, HTML, Image as IPImage, Markdown
import json, os, time

output_dir = os.path.join(os.path.dirname(os.getcwd()), "examples", "outputs", "google_maps")
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")
print(f"Maps key: {'set' if gm._get_api_key() else 'MISSING'}")

---
## 1. Geocoding

Convert addresses to coordinates and back.

In [ ]:
# Forward geocode
loc = gm.geocode("100 S State St, Salt Lake City, UT")
print(f"Address: {loc['formatted_address']}")
print(f"Coords: {loc['lat']}, {loc['lon']}")
print(f"Accuracy: {loc['location_type']}")

# Reverse geocode
rev = gm.reverse_geocode(loc['lon'], loc['lat'])
print(f"\nReverse: {rev['formatted_address']}")

# Address validation
val = gm.validate_address("4200 olympic way slc ut")
if val:
    print(f"\nValidated: {val['formatted_address']}")
    print(f"Complete: {val['verdict']['address_complete']}")
    if val.get('usps_data'):
        print(f"USPS: {val['usps_data']['standardized_address'].get('firstAddressLine')}")

---
## 2. Street View

Fetch Street View imagery — single frames, cardinal directions, stitched panoramas.

In [ ]:
# Check coverage
meta = gm.streetview_metadata(loc['lon'], loc['lat'])
print(f"Status: {meta['status']}, Date: {meta.get('date')}")

# Single image
img = gm.streetview_image(loc['lon'], loc['lat'], heading=120)
if img:
    display(IPImage(data=img, width=600))
    print(f"Single frame: {len(img):,} bytes")

In [ ]:
# 360° stitched panorama
pano = gm.streetview_panorama(loc['lon'], loc['lat'], heading=120, fov=360)
if pano:
    display(IPImage(data=pano, width=900))
    print(f"360° panorama: {len(pano):,} bytes")

In [ ]:
# HTML panel with cardinal directions
html = gm.streetview_html(loc['lon'], loc['lat'])
if html:
    display(HTML(html))

---
## 3. Places

Search for places near a location.

In [ ]:
# Text search
places = gm.search_places("coffee shop", lat=loc['lat'], lon=loc['lon'], radius=2000)
print(f"Found {len(places)} coffee shops:\n")
for p in places[:5]:
    rating = f" ({p['rating']}★)" if p.get('rating') else ""
    print(f"  {p['display_name']}{rating}: {p['address']}")

# Nearby search
parks = gm.search_nearby(loc['lat'], loc['lon'], radius=3000, included_types=["park"])
print(f"\nFound {len(parks)} parks nearby")
for p in parks[:3]:
    print(f"  {p['display_name']}: {p['address']}")

---
## 4. Environment

Elevation, air quality, solar potential, and timezone.

In [ ]:
# Elevation
elev = gm.get_elevation(loc['lon'], loc['lat'])
print(f"Elevation: {elev:.0f} m ({elev * 3.28084:.0f} ft)")

# Elevation profile
profile = gm.get_elevation_along_path(
    [(-111.89, 40.77), (-111.80, 40.68)],  # SLC downtown to Wasatch foothills
    samples=10,
)
print(f"\nElevation profile ({len(profile)} points):")
for p in profile:
    print(f"  ({p['lat']:.4f}, {p['lon']:.4f}): {p['elevation']:.0f} m")

In [ ]:
# Air quality
aq = gm.get_air_quality(loc['lon'], loc['lat'])
if aq:
    print(f"AQI: {aq['aqi']} — {aq['category']}")
    print(f"Dominant pollutant: {aq['dominant_pollutant']}")

# Solar potential
solar = gm.get_solar_insights(loc['lon'], loc['lat'])
if solar:
    print(f"\nSolar potential:")
    print(f"  Max panels: {solar['max_panels']}")
    print(f"  Annual output: {solar['max_annual_kwh']:,.0f} kWh")
    print(f"  Roof area: {solar['roof_area_m2']:.0f} m²")
# Show static map of the location
static_map = gm.get_static_map(loc['lon'], loc['lat'],zoom=16,markers=[(loc['lon'],loc['lat'])])
if static_map:
    display(IPImage(data=static_map, width=600))

# Timezone
tz = gm.get_timezone(loc['lon'], loc['lat'])
if tz:
    print(f"\nTimezone: {tz['timezone_id']} ({tz['timezone_name']})")

---
## 5. AI Analysis (Gemini)

Use Gemini to interpret and label Street View imagery. Requires `GEMINI_API_KEY`.

In [ ]:
display(IPImage(data=pano, width=900))
print('Intpretting image...')
# Interpret a panorama
result = gm.interpret_image(
    pano,
    context=f"{loc['formatted_address']}. Downtown Salt Lake City.",
)
print('Finished intpretting image')

display(Markdown(result['description']))
display(Markdown(result['object_counts']))

In [ ]:
# Label objects with bounding boxes
labeled = gm.label_streetview(
    loc['lon'], loc['lat'],
    heading=120, fov=360,
    max_labels=20,
)
if labeled:
    display(IPImage(data=labeled['image'], width=900))
    print(f"Detected {len(labeled['detections'])} objects")
    display(Markdown(labeled['summary']))

---
## 6. Semantic Segmentation (SegFormer)

Pixel-level land cover classification. Requires `pip install geeViz[segmentation]`.

In [ ]:
# Segment the panorama with SegFormer B4
try:
    t0 = time.time()
    seg = gm.segment_image(pano, model_variant="b4", broad_categories=False)
    print(f"Segmented in {time.time()-t0:.1f}s")
    display(IPImage(data=seg['colored_image'], width=900))
    display(Markdown(seg['summary']))
except ImportError:
    print("SegFormer requires: pip install geeViz[segmentation]")

In [ ]:
# Segment with broad land cover categories
try:
    seg_broad = gm.segment_image(pano, model_variant="b4", broad_categories=True)
    display(IPImage(data=seg_broad['colored_image'], width=900))
    display(Markdown(seg_broad['summary']))
except ImportError:
    print("SegFormer requires: pip install geeViz[segmentation]")

---
## 7. Roads & Static Maps

In [ ]:
# Nearest roads
roads = gm.nearest_roads(loc['lon'], loc['lat'])
print(f"Nearest roads: {len(roads)} segments")
for r in roads:
    print(f"  ({r['lat']:.5f}, {r['lon']:.5f})")

# Static map
static = gm.get_static_map(
    loc['lon'], loc['lat'],
    zoom=16, size="640x400", maptype="hybrid",
    markers=[(loc['lon'], loc['lat'])],
)
if static:
    display(IPImage(data=static, width=640))
    print(f"Static map: {len(static):,} bytes")

---
## Summary

| Category | Functions | API Key |
|---|---|---|
| Geocoding | `geocode`, `reverse_geocode`, `validate_address` | `GOOGLE_MAPS_PLATFORM_API_KEY` |
| Places | `search_places`, `search_nearby`, `get_place_photo` | `GOOGLE_MAPS_PLATFORM_API_KEY` |
| Street View | `streetview_metadata`, `streetview_image`, `streetview_panorama`, `streetview_html` | `GOOGLE_MAPS_PLATFORM_API_KEY` |
| AI Analysis | `interpret_image`, `label_streetview` | `GEMINI_API_KEY` |
| Segmentation | `segment_image`, `segment_streetview` | None (local model) |
| Elevation | `get_elevation`, `get_elevations`, `get_elevation_along_path` | `GOOGLE_MAPS_PLATFORM_API_KEY` |
| Environment | `get_air_quality`, `get_solar_insights`, `get_timezone` | `GOOGLE_MAPS_PLATFORM_API_KEY` |
| Maps & Roads | `get_static_map`, `snap_to_roads`, `nearest_roads` | `GOOGLE_MAPS_PLATFORM_API_KEY` |